# 03 - Embeddings e indexacion en Qdrant

Objetivo: unir documentos, chunking, embeddings de Ollama e indexacion en Qdrant.


In [1]:
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv(Path("..").resolve() / ".env")

OLLAMA_PORT = os.getenv("OLLAMA_PORT", "11434")
QDRANT_PORT = os.getenv("QDRANT_PORT", "6333")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embeddinggemma")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "frasohome_docs")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))

DOCS_DIR = Path("..").resolve() / "docs"
OLLAMA_BASE_URL = f"http://127.0.0.1:{OLLAMA_PORT}"
QDRANT_URL = f"http://127.0.0.1:{QDRANT_PORT}"


In [2]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

for idx, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = idx

print(f"Documentos: {len(documents)}")
print(f"Chunks: {len(chunks)}")


Documentos: 2
Chunks: 9


In [3]:
embedding_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
texts = [chunk.page_content for chunk in chunks]
vectors = embedding_model.embed_documents(texts)
vector_size = len(vectors[0])

client = QdrantClient(url=QDRANT_URL)
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

points = []
for chunk, vector in zip(chunks, vectors):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "source": chunk.metadata.get("source", "desconocido"),
                "chunk_id": chunk.metadata["chunk_id"],
                "text": chunk.page_content,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.get_collection(COLLECTION_NAME)


C:\Users\anton\AppData\Local\Temp\ipykernel_22064\1242869493.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=9, segments_count=4, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None),

In [4]:
question = "Que debo hacer si faltan tornillos en la caja?"
query_vector = embedding_model.embed_query(question)
hits = client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=3).points

for hit in hits:
    print("=" * 80)
    print({
        "score": round(hit.score, 4),
        "source": hit.payload.get("source"),
        "chunk_id": hit.payload.get("chunk_id"),
    })
    print(hit.payload.get("text", "")[:500])


{'score': 0.5773, 'source': 'C:\\Repos\\rag-local-lab\\docs\\manual_mesa_oslo.md', 'chunk_id': 2}
## 5. Importante
**No aprietes totalmente los tornillos en el primer paso.**  
FrasoHome recomienda un apriete parcial inicial para facilitar la alineación del conjunto.

## 6. Si faltan piezas o tornillos
Si durante la apertura de la caja faltan tornillos, patas, travesaños o la llave Allen:
- no fuerces el montaje,
- conserva el embalaje,
- abre una incidencia con el número de pedido,
- indica la referencia `FH-MOSLO-140`,
- adjunta una foto del contenido recibido.
{'score': 0.4372, 'source': 'C:\\Repos\\rag-local-lab\\docs\\manual_mesa_oslo.md', 'chunk_id': 1}
## 3. Preparación previa
Monta la mesa sobre una superficie limpia y protegida. No arrastres el tablero sobre suelos rugosos. Se recomienda dejar todas las piezas presentadas antes de apretar por completo la tornillería.

## 4. Secuencia de montaje
1. Coloca el tablero boca abajo sobre una manta o cartón protector.
2. Presenta las